# ========================================
# 02_LLM_Extraction.ipynb
# ========================================

In [ ]:
# ========================================
# 02_LLM_Extraction_Upgraded.ipynb
# ========================================

# 1. Install (if not already)
# pip install transformers datasets torch

# 2. Imports
import pandas as pd
import torch
import json
import os
from transformers import T5Tokenizer, T5ForConditionalGeneration
from tqdm import tqdm

# 3. Load Cleaned Data
df = pd.read_csv('/Users/ronitgandhi/Portfolio Projects/job_postings_insights/data/cleaned_job_postings.csv')

# 4. Load flan-t5-large Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

model_name = "google/flan-t5-large"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

# 5. Force JSON Prompt
def create_prompt(text):
    return f"""
You are an information extraction system.

From the following job description, extract the following structured information and respond ONLY in valid JSON format.

Required JSON Keys:
- Job Title
- Skills (comma separated list)
- Salary Range
- Location
- Remote (Yes or No)

Example:
{{
"Job Title": "Data Scientist",
"Skills": ["Python", "SQL", "Machine Learning"],
"Salary Range": "$80,000-$120,000",
"Location": "New York, NY",
"Remote": "Yes"
}}

Description:
{text}
"""

# 6. Extraction Function
def extract_fields(text):
    if pd.isnull(text) or len(text.strip()) < 20:
        return {}
    # Limit to first 700 characters
    text = text[:700]
    prompt = create_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_length=256)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    try:
        result = json.loads(decoded)
        return result
    except Exception as e:
        print("JSON parsing error:", e)
        return {}

# 7. Apply to Dataset (Small sample first)
sampled_df = df.sample(200, random_state=42)  # 👈 200 rows first
results = []

for desc in tqdm(sampled_df['description']):
    res = extract_fields(desc)
    results.append(res)

# 8. Combine with Original
extracted_df = pd.DataFrame(results)
combined_df = pd.concat([sampled_df.reset_index(drop=True), extracted_df], axis=1)

# 9. Save Extracted Data
os.makedirs('outputs/extracted_data/', exist_ok=True)
combined_df.to_csv('/Users/ronitgandhi/Portfolio Projects/job_postings_insights/outputs/extracted_data/extracted_fields_sample_upgraded.csv', index=False)

print("\n✅ Clean JSON-based extracted fields saved to outputs/extracted_data/extracted_fields_sample_upgraded.csv")


# ========================================
# 03_Postprocess_Extracted_Fields.ipynb
# ========================================

In [9]:

import pandas as pd
import numpy as np
import re

# 1. Load sampled data
df = pd.read_csv('/Users/ronitgandhi/Portfolio Projects/job_postings_insights/data/cleaned_job_postings.csv')

# 2. Skills Extraction
def extract_skills(description):
    keywords = [
        'Python', 'SQL', 'Machine Learning', 'Deep Learning', 'AWS', 'Azure',
        'Data Analysis', 'Statistics', 'NLP', 'TensorFlow', 'PyTorch', 'Excel', 
        'Leadership', 'Communication', 'Project Management', 'R', 'Tableau', 'BigQuery'
    ]
    if pd.isnull(description):
        return np.nan
    found_skills = []
    for skill in keywords:
        if skill.lower() in description.lower():
            found_skills.append(skill)
    return found_skills if found_skills else np.nan

df['Skills_Extracted'] = df['description'].apply(extract_skills)

# 3. Remote Detection
def detect_remote(description):
    if pd.isnull(description):
        return "No"
    desc = description.lower()
    if 'remote' in desc or 'work from home' in desc or 'hybrid' in desc:
        return "Yes"
    return "No"

df['Remote_Allowed_Detected'] = df['description'].apply(detect_remote)

# 4. Salary Building
def build_salary(min_salary, max_salary):
    if not np.isnan(min_salary) and not np.isnan(max_salary):
        return f"${int(min_salary)} - ${int(max_salary)}"
    return np.nan

df['Salary_Range_Built'] = [
    build_salary(mins, maxs) for mins, maxs in zip(df['min_salary'], df['max_salary'])
]

# 5. Final structured DataFrame
final_df = pd.DataFrame({
    'Job ID': df['job_id'],
    'Company Name': df['company_name'],
    'Job Title': df['title'],
    'Skills': df['Skills_Extracted'],
    'Salary Range': df['Salary_Range_Built'],
    'Location': df['location'],
    'Remote Allowed': df['Remote_Allowed_Detected'],
    'Original Description': df['description']
})

# 6. Save final structured data
final_df.to_csv('/Users/ronitgandhi/Portfolio Projects/job_postings_insights/outputs/extracted_data/final_structured_job_postings.csv', index=False)

print("\n✅ Final structured dataset saved to outputs/extracted_data/final_structured_job_postings.csv")



✅ Final structured dataset saved to outputs/extracted_data/final_structured_job_postings.csv
